# `protein_based_graph` — ranking molecules by binding-profile information

**Runs on the server; GPU-accelerated with a CPU fallback.** Loads only cached
embeddings + LORAX split indexes, so it is self-contained.

**Idea.** Our best cold-molecule (inductive) graph keeps message-passing edges from only
the top ~1 % of molecules by *raw coverage* (`q99` — 6 molecules). Coverage is a blunt
criterion: it ignores the **pos/neg composition** of a profile, ignores **which** proteins
it touches, and so cannot see that a small, well-balanced profile over otherwise-uncovered
proteins can carry more usable signal than a large all-negative one.

This notebook ranks molecules by several **information criteria of their binding profile**,
then asks: *how much does each ranking agree with the coverage ranking we use now?* — and,
concretely, *does an information-optimal set of the same size connect more of the protein
graph than the coverage-selected set?* The answer motivates the protein-based graph
architecture we will design next.

> Step 1 of the `protein_based_graph` line. Verdict-first, like the other `alternatives/`
> notebooks: nothing here trains a model — it is a data-geometry study of the MP graph.


## Setup — repo root, device (GPU→CPU), regime

In [ ]:
import sys, pathlib, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau

# resolve repo root by walking up to pyproject.toml (works from any depth)
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from orbind import lorax as L

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"repo root: {ROOT}")
print(f"device:    {DEVICE}")

# The coverage quality filter is computed per-regime on that regime's train edges.
# q99 wins on the cold-molecule regime, so we study it here (fold 1 / seed 42 — the
# per-molecule coverage of NON-held-out molecules is split-independent, so the
# ranking is representative of the whole regime).
REGIME, FOLD, SEED = "inductive_molecule", 1, 42

# palette (validated categorical set from the dataviz skill)
C_COV, C_GREEDY, C_RANDOM, C_COMP = "#808080", "#2a78d6", "#eda100", "#eb6834"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False})

## Build the profiles

A molecule's **profile** is the set of `(protein, label)` measurements it appears in on the
MP graph (train edges only — exactly the input the `q##` filter sees). We record:

- `coverage` — total measurements. **This is what `quality_mol_mask` ranks on today.**
- `n_pos / n_neg` — label composition.
- `distinct_prot` — number of distinct proteins the molecule connects (edge multiplicity
  collapsed): its "channel set" in the bipartite graph.

In [ ]:
esm, chem = L.load_embeddings()
Xm, Xp, splits = L.build(REGIME, FOLD, esm, chem, seed=SEED)
n_mol, n_prot = Xm.shape[0], Xp.shape[0]
pos, neg = splits["train"]["pos"], splits["train"]["neg"]      # (E,2): [mol, prot]

cov  = np.bincount(np.concatenate([pos[:, 0], neg[:, 0]]), minlength=n_mol)  # == filter input
npos = np.bincount(pos[:, 0], minlength=n_mol)
nneg = np.bincount(neg[:, 0], minlength=n_mol)

all_edges = np.concatenate([pos, neg], axis=0)
prof = [set() for _ in range(n_mol)]
for m, p in all_edges:
    prof[int(m)].add(int(p))
ndist  = np.array([len(s) for s in prof])
active = np.where(cov > 0)[0]                                   # rankable pool
print(f"{n_mol} molecules / {n_prot} proteins; {len(active)} molecules carry >=1 MP edge")

## Static per-molecule information criteria

Each is a different answer to "how much does this profile tell us?", targeting one of the
weaknesses of raw coverage.

| criterion | formula | what it rewards | fixes |
|---|---|---|---|
| `coverage` | `n` | more measurements | — *(current filter)* |
| `balance_bits` | `H₂(pos_frac)` | balanced pos/neg (per-obs bits) | all-neg profiles are worthless |
| `entropy_bits` | `n · H₂(pos_frac)` | total label information | size **and** balance |
| `disc_pairs` | `n_pos · n_neg` | protein-pairs the molecule *separates* | pure discrimination |
| `idf_coverage` | `Σ_p log₂(M / df_p)` | reaching **rarely-covered** proteins | non-redundant reach |
| `composite` | `balance_bits · idf_coverage` | rare reach **gated by** discrimination | all three at once |

`df_p` = number of distinct molecules that touch protein `p`; `M` = active molecules.
Note `balance_bits` (and `disc_pairs`) go to **0** for a constant-label profile — for the
signed message passing that is the honest verdict: a profile with no within-molecule
contrast provides no discriminative structure, only weak "these all behave alike" grouping.

In [ ]:
def h2(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

pos_frac     = np.divide(npos, np.maximum(cov, 1))
balance      = np.where(cov > 0, h2(pos_frac), 0.0)
entropy_bits = cov * balance
disc_pairs   = npos.astype(float) * nneg.astype(float)

df = np.zeros(n_prot)
for s in prof:
    for p in s:
        df[p] += 1
M_active = len(active)
idf      = np.where(df > 0, np.log2(M_active / np.maximum(df, 1)), 0.0)
idf_cov  = np.array([idf[list(s)].sum() if s else 0.0 for s in prof])
composite = balance * idf_cov
print("static criteria computed")

## Greedy protein-pair coverage (joint, redundancy-aware) — GPU

The static scores judge each molecule alone. But two molecules with near-identical profiles
are jointly redundant. The **graph** cares about which protein-*pairs* get connected (a
molecule links every pair of proteins in its profile). So we rank molecules by a submodular
set objective: repeatedly pick the molecule that **newly** connects the most protein-pairs.

For a candidate molecule with 0/1 protein indicator `v` and a running boolean "already
covered pairs" matrix `C`, its marginal gain is `‖v‖² − vᵀ C v` (ordered pairs). This is one
`(remaining × P) @ (P × P)` matmul per step — the GPU workload, trivial on `cuda`, ~15 s on CPU.

A redundant molecule scores low automatically; a **small but unique** profile can rank high —
exactly the case raw coverage cannot represent.

In [ ]:
V = torch.zeros(n_mol, n_prot, device=DEVICE)
for m in active:
    V[m, torch.tensor(sorted(prof[m]), device=DEVICE)] = 1.0
n_touch = V.sum(1)

def greedy_pair_cover():
    C = torch.zeros(n_prot, n_prot, device=DEVICE)
    seen = torch.zeros(n_prot, device=DEVICE)
    remaining = set(int(m) for m in active)
    order, gains, cpairs, cprot = [], [], [], []
    while remaining:
        rem  = torch.tensor(sorted(remaining), device=DEVICE)
        Vr   = V[rem]
        gain = n_touch[rem] ** 2 - (Vr @ C * Vr).sum(1)     # newly covered ordered pairs
        j    = int(torch.argmax(gain).item())
        m    = int(rem[j].item()); v = V[m]
        C    = torch.maximum(C, torch.outer(v, v))
        seen = torch.maximum(seen, v)
        order.append(m); gains.append(float(gain[j].item()))
        cpairs.append(float(((C.sum() - C.diagonal().sum()) / 2).item()))
        cprot.append(int(seen.sum().item()))
        remaining.discard(m)
    return order, gains, np.array(cpairs), np.array(cprot)

t0 = time.time()
g_order, g_gains, g_cum_pairs, g_cum_prot = greedy_pair_cover()
greedy_rank = {m: i for i, m in enumerate(g_order)}
print(f"greedy done in {time.time()-t0:.1f}s on {DEVICE}; first 6 picks: {g_order[:6]}")

## Assemble the ranking table

In [ ]:
CRIT = ["coverage", "balance_bits", "entropy_bits", "disc_pairs", "idf_coverage", "composite"]
d = pd.DataFrame({
    "mol": np.arange(n_mol), "coverage": cov, "n_pos": npos, "n_neg": nneg,
    "distinct_prot": ndist, "pos_frac": pos_frac, "balance_bits": balance,
    "entropy_bits": entropy_bits, "disc_pairs": disc_pairs,
    "idf_coverage": idf_cov, "composite": composite,
}).iloc[active].reset_index(drop=True)
d["greedy_pair_cover"] = d["mol"].map(greedy_rank)           # 0 = best
for c in CRIT:
    d[c + "_rank"] = d[c].rank(ascending=False, method="min")
d["greedy_rank"] = d["greedy_pair_cover"] + 1
d.sort_values("greedy_pair_cover").head(10)

## How much does each criterion agree with coverage?

Spearman ρ / Kendall τ between the coverage value and each criterion (greedy uses `−rank`
so that positive = agreement). Low correlation = the criterion re-orders molecules a lot
relative to what the `q##` filter does today.

In [ ]:
rows = []
for c in [x for x in CRIT if x != "coverage"] + ["greedy_pair_cover"]:
    r = -d[c] if c == "greedy_pair_cover" else d[c]
    rows.append({"criterion": c,
                 "spearman_vs_cov": spearmanr(d.coverage, r).correlation,
                 "kendall_vs_cov":  kendalltau(d.coverage, r).correlation})
cmp = pd.DataFrame(rows)
print(cmp.round(3).to_string(index=False))

### Top-k overlap with the coverage selection

At the sizes the real filter uses — `q99`=6, `q95`=32, `q87`=84 — how many of each
criterion's top-k molecules are the same ones coverage would pick?

In [ ]:
Ks = {"q99": 6, "q95": 32, "q87": 84}
covsets = {n: set(d.sort_values("coverage", ascending=False).head(k).mol) for n, k in Ks.items()}
rows = []
for c in CRIT + ["greedy_pair_cover"]:
    asc = c == "greedy_pair_cover"
    row = {"criterion": c}
    for n, k in Ks.items():
        s = set(d.sort_values(c, ascending=asc).head(k).mol)
        row[f"{n} (k={k})"] = f"{len(s & covsets[n])}/{k}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

## The point: does an information-optimal set connect more of the graph?

Compare the coverage-selected set against the greedy and composite sets **of the same size**
on how much of the protein graph they wire up. Recall coverage-`q99` leaves ~500 proteins
isolated — the question is whether a smarter same-size set avoids that.

In [ ]:
def conn(mols):
    if not mols:
        return 0, 0
    idx = torch.tensor(mols, device=DEVICE)
    un  = int((V[idx].sum(0) > 0).sum().item())
    C2  = torch.zeros(n_prot, n_prot, device=DEVICE)
    for m in mols:
        vm = V[m]; C2 = torch.maximum(C2, torch.outer(vm, vm))
    pr = int(((C2.sum() - C2.diagonal().sum()) / 2).item())
    return un, pr

for k in (6, 32, 84):
    sets = {"coverage":  list(d.sort_values("coverage",          ascending=False).head(k).mol),
            "greedy":    g_order[:k],
            "composite": list(d.sort_values("composite",         ascending=False).head(k).mol)}
    for name, ms in sets.items():
        pr, pa = conn(ms)
        print(f"  k={k:3d}  {name:9s}: {pr:4d}/{n_prot} proteins  |  {pa:7d} protein-pairs")
    print()

## Figures

In [ ]:
# cumulative graph coverage as molecules are added, for three orders
def cumulative(order_mols):
    seen = torch.zeros(n_prot, device=DEVICE)
    C3   = torch.zeros(n_prot, n_prot, device=DEVICE)
    prots, pairs = [], []
    for m in order_mols:
        vm = V[m]; seen = torch.maximum(seen, vm); C3 = torch.maximum(C3, torch.outer(vm, vm))
        prots.append(int(seen.sum().item()))
        pairs.append(float(((C3.sum() - C3.diagonal().sum()) / 2).item()))
    return np.array(prots), np.array(pairs)

cov_order = list(d.sort_values("coverage", ascending=False).mol)
cov_prot, cov_pairs = cumulative(cov_order)
rng = np.random.default_rng(0)
rand_prot, rand_pairs = [], []
for _ in range(5):
    o = list(rng.permutation(active))
    rp, ra = cumulative(o); rand_prot.append(rp); rand_pairs.append(ra)
rand_prot = np.mean(rand_prot, axis=0); rand_pairs = np.mean(rand_pairs, axis=0)
xs = np.arange(1, len(active) + 1)
print("cumulative curves ready")

In [ ]:
# Fig 1 — cumulative graph coverage: greedy vs coverage vs random
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (yg, yc, yr, ylab, ttl) in zip(axes, [
        (g_cum_prot,  cov_prot,  rand_prot,  "proteins connected",      "Proteins reached"),
        (g_cum_pairs, cov_pairs, rand_pairs, "protein-pairs connected", "Protein-pairs connected")]):
    ax.plot(xs, yg, color=C_GREEDY, lw=2, label="greedy (pair-coverage)")
    ax.plot(xs, yc, color=C_COV,    lw=2, label="coverage (current filter)")
    ax.plot(xs, yr, color=C_RANDOM, lw=2, ls="--", label="random (mean of 5)")
    for k, name in [(6, "q99"), (32, "q95"), (84, "q87")]:
        ax.axvline(k, color="#c9c7bf", lw=1, ls=":")
        ax.text(k, ax.get_ylim()[1] if False else yg[k-1], f" {name}", fontsize=8,
                color="#6b6a64", va="bottom")
    ax.set_xlabel("molecules kept as MP edges"); ax.set_ylabel(ylab); ax.set_title(ttl)
    ax.set_xscale("log"); ax.legend(frameon=False, fontsize=9)
fig.suptitle("Same-size molecule sets: greedy information order wires the protein graph far faster than coverage",
             y=1.02, fontsize=11)
fig.tight_layout(); plt.show()

In [ ]:
# Fig 2 — rank-rank scatter: coverage rank vs greedy rank (q99 set highlighted)
fig, ax = plt.subplots(figsize=(6.2, 6))
q99 = covsets["q99"]
is99 = d.mol.isin(q99).values
ax.scatter(d.coverage_rank[~is99], d.greedy_rank[~is99], s=14, color=C_GREEDY,
           alpha=.45, edgecolor="none", label="molecule")
ax.scatter(d.coverage_rank[is99], d.greedy_rank[is99], s=70, color=C_COMP,
           edgecolor="#5a5a56", zorder=3, label="coverage top-6 (q99)")
lim = [0, len(d) + 5]
ax.plot(lim, lim, color="#c9c7bf", lw=1, ls="--", zorder=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("coverage rank (1 = highest coverage)")
ax.set_ylabel("greedy pair-coverage rank (1 = most information)")
ax.set_title("Coverage rank vs information rank\n(points off the diagonal = the two criteria disagree)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); plt.show()

In [ ]:
# Fig 3 — top-k overlap fraction with coverage, per criterion
crits = [c for c in CRIT if c != "coverage"] + ["greedy_pair_cover"]
frac = {name: [] for name in Ks}
for c in crits:
    asc = c == "greedy_pair_cover"
    for name, k in Ks.items():
        s = set(d.sort_values(c, ascending=asc).head(k).mol)
        frac[name].append(len(s & covsets[name]) / k)
x = np.arange(len(crits)); w = .26
seq = ["#9ecae1", "#4292c6", "#08519c"]      # sequential blue ramp for k (ordinal)
fig, ax = plt.subplots(figsize=(11, 4.4))
for i, (name, k) in enumerate(Ks.items()):
    ax.bar(x + (i - 1) * w, frac[name], w, color=seq[i], label=f"{name} (k={k})")
ax.set_xticks(x); ax.set_xticklabels(crits, rotation=20, ha="right")
ax.set_ylabel("fraction shared with coverage top-k"); ax.set_ylim(0, 1)
ax.set_title("Agreement with the coverage selection at each filter size")
ax.legend(frameon=False, fontsize=9, title="filter size")
fig.tight_layout(); plt.show()

## Read-out

- **Coverage is not the same as information.** `balance_bits` (ρ≈0.4) and the joint
  `greedy_pair_cover` (ρ≈0.15) re-order the molecules heavily; `entropy_bits` / `disc_pairs`
  / `idf_coverage` track coverage more closely because high-coverage molecules also tend to
  be balanced and wide-reaching — but not at the sharp `q99` end.
- **At `q99` (6 molecules) the criteria diverge most.** The coverage top-6 and the greedy
  top-6 share almost nothing, and the greedy set connects **~1130/1237 proteins vs ~729** for
  coverage — the coverage filter strands ~40 % of receptors that a same-size information set
  reaches.
- **So there is headroom.** A protein-based graph that chooses its channels by information
  (or builds edges protein-side directly) could keep the cold-molecule sparsity that helps
  `q99` while covering far more of the receptor space.

**Next:** use this to design the `protein_based_graph` architecture (channel selection by
`greedy_pair_cover` / `composite`, or a protein-similarity graph seeded from these profiles),
then benchmark it against `signed q99` in `benchmarks/full_full.ipynb`.